DeepEval Agentic Metrics Learning

Agentic metrics are used to evaluate AI applications that behave like agents.

An agent usually does:
user request
→ plans or decides steps
→ selects tools
→ calls tools
→ uses tool results
→ gives final answer

Agentic metrics check:
- Did the agent complete the task?
- Did the agent choose the correct tool?
- Did the agent pass correct arguments?
- Did the agent follow the plan?
- Did the agent avoid unnecessary steps?

This notebook is created separately for learning Agentic Metrics.

In [1]:
!pip install deepeval groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.7/567.7 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 1.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


In [2]:
import os
from google.colab import userdata
from groq import Groq

from deepeval.models import DeepEvalBaseLLM

In [3]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


class GroqDeepEvalModel(DeepEvalBaseLLM):

    def __init__(self, model_name="openai/gpt-oss-20b"):
        self.model_name = model_name
        self.client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def load_model(self):
        return self.client

    def generate(self, prompt: str, **kwargs) -> str:
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "system",
                    "content": "Return only valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        return response.choices[0].message.content

    async def a_generate(self, prompt: str, **kwargs) -> str:
        return self.generate(prompt, **kwargs)

    def get_model_name(self):
        return self.model_name


groq_model = GroqDeepEvalModel()

print("Groq evaluator connected.")

Groq evaluator connected.


In [12]:
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.evaluate import ErrorConfig, AsyncConfig
from deepeval.tracing import observe, update_current_trace
from deepeval.metrics import TaskCompletionMetric, StepEfficiencyMetric

TaskCompletionMetric

TaskCompletionMetric checks whether an AI agent successfully completed the user’s task.

It focuses on the final outcome of the agent’s work.

If the agent’s response fully satisfies the user request, it passes.

If the agent gives only partial help, unclear help, or does not complete the requested task, it fails.

In [5]:
task_completion_metric = TaskCompletionMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Task Completion metric created.")

Task Completion metric created.


In [6]:
agent_goldens = [
    Golden(input="Help the customer track their order."),
    Golden(input="Help the customer understand the refund policy."),
    Golden(input="Help the customer change their delivery address before shipment.")
]

agent_dataset = EvaluationDataset(goldens=agent_goldens)

print("Agent task dataset created.")

Agent task dataset created.


In [7]:
def track_order_tool():
    return "Order can be tracked using the tracking link sent to the registered email or mobile number."


def refund_policy_tool():
    return "Refund is allowed within 15 days if the item is unused and in original condition."


def change_address_tool():
    return "Delivery address can be changed before shipment by contacting customer support."

In [10]:
@observe()
def customer_support_agent(user_task):
    if "track" in user_task.lower():
        answer = track_order_tool()

    elif "refund" in user_task.lower():
        answer = refund_policy_tool()

    elif "delivery address" in user_task.lower() or "address" in user_task.lower():
        answer = change_address_tool()

    else:
        answer = "I could not identify the correct support action."

    update_current_trace(
        input=user_task,
        output=answer
    )

    return answer

In [11]:
for golden in agent_dataset.evals_iterator(
    metrics=[task_completion_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Agentic Task Completion evaluation completed.")

Output()

Task: Help the customer track their order.

Agent answer: Order can be tracked using the tracking link sent to the registered email or mobile number.

--------------------------------------------------------------------------------

Task: Help the customer understand the refund policy.

...

--------------------------------------------------------------------------------

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_3                                                                                                 │
│  ├──   Input:            Help the customer change their delivery address before shipment.                       │
│  │     Actual Output:    Delivery address can be changed before shipment by contacting customer support.        │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric          ┃ Score ┃ Threshold ┃ Reason                                                     │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Task Completion │ 0.35  │ 0.60      │ The response only informs the customer that the address    │
│              │                 │       │           │ can be changed by contacting support, but it does not      │
│              │                 │       │           │ provide any direct assistance, steps, or options to        │
│              │                 │       │           │ actually change the address within the system. Therefore   │
│              │                 │       │           │ it partially addresses the task but falls short of fully   │
│              │                 │       │           │ helping the customer.                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                    ┃ Average Score          ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Task Completion           │ 0.58                   │ 66.67% | passed=2 | failed=1                  │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=588498;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.27s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 66.67% | Passed: 2 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Agentic Task Completion evaluation completed.


StepEfficiencyMetric

StepEfficiencyMetric checks whether an AI agent completed the task using useful and necessary steps.

It focuses on the agent’s execution path.

If the agent uses direct and relevant steps, it passes.

If the agent uses unnecessary, repeated, or unrelated steps, it fails.

In [13]:
step_efficiency_metric = StepEfficiencyMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Step Efficiency metric created.")

Step Efficiency metric created.


In [14]:
for golden in agent_dataset.evals_iterator(
    metrics=[step_efficiency_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Agentic Step Efficiency evaluation completed.")

Output()

Task: Help the customer track their order.

Agent answer: Order can be tracked using the tracking link sent to the registered email or mobile number.

--------------------------------------------------------------------------------

Task: Help the customer understand the refund policy.

...

--------------------------------------------------------------------------------

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                    ┃ Average Score         ┃ Pass Rate                                      ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Step Efficiency           │ 1.00                  │ 100.00% | passed=3 | failed=0                  │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=762195;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.12s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Agentic Step Efficiency evaluation completed.


ToolCorrectnessMetric

ToolCorrectnessMetric checks whether an AI agent selected the correct tool for the given task.

It compares the tools actually used by the agent with the tools expected for that task.

If the agent calls the correct tool, it passes.

If the agent calls the wrong tool, misses a required tool, or uses an unnecessary tool, it fails.